# Run and evaluate the Energy Advisor
Run the setup notebooks first. The questions below cover scheduling, thermostat guidance, history-based advice, savings, and solar-aware pool-pump planning.

In [8]:
from IPython.display import Markdown, display
from agent import build_energy_advisor

forecast_tools = ["get_weather_forecast", "get_electricity_prices", "get_solar_generation", "retrieve_energy_tips"]
scenarios = [
    {"id": "ev-solar-tomorrow", "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?", "expected_tools": ["optimize_device_schedule", *forecast_tools], "expected_response": ["EV Charger", "estimated savings", "forecast solar", "Knowledge base:"]},
    {"id": "dishwasher-off-peak", "question": "How much can I save by running my dishwasher during off-peak hours?", "expected_tools": ["optimize_device_schedule", *forecast_tools], "expected_response": ["Dishwasher", "estimated savings", "18:00 start", "Knowledge base:"]},
    {"id": "pool-pump-weekly", "question": "What's the best time to run my pool pump this week based on the weather forecast?", "expected_tools": ["optimize_device_schedule_this_week", *forecast_tools], "expected_response": ["Pool Pump", "estimated savings", "Forecast:", "Knowledge base:"]},
    {"id": "thermostat-price-spike", "question": "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?", "expected_tools": forecast_tools, "expected_response": ["setpoint", "comfort and health needs", "forecast averages", "Knowledge base:"]},
    {"id": "usage-history-actions", "question": "Suggest three ways I can reduce energy use based on my usage history.", "expected_tools": [*forecast_tools, "summarize_usage_history"], "expected_response": ["highest measured load", "Three practical actions", "Knowledge base:"]},
    {"id": "washer-schedule", "question": "Schedule my washer for the lowest-cost time tomorrow.", "expected_tools": ["optimize_device_schedule", *forecast_tools], "expected_response": ["Washer", "estimated energy cost", "estimated savings", "Knowledge base:"]},
    {"id": "hvac-weather-guidance", "question": "What thermostat setting should I use tomorrow afternoon to save energy without sacrificing comfort?", "expected_tools": forecast_tools, "expected_response": ["setpoint", "comfort and health needs", "forecast averages", "Knowledge base:"]},
    {"id": "battery-roi", "question": "Estimate ROI and payback for a home battery.", "inputs": {"investment_cost_usd": 1200, "annual_savings_usd": 400, "incentives_usd": 200}, "expected_tools": [*forecast_tools, "summarize_usage_history", "calculate_roi"], "expected_response": ["net cost $1000.00", "simple payback 2.5 years", "excludes financing", "Knowledge base:"]},
    {"id": "history-based-reduction", "question": "Based on my recent energy usage history, give me practical steps to reduce consumption.", "expected_tools": [*forecast_tools, "summarize_usage_history"], "expected_response": ["highest measured load", "Three practical actions", "flexible loads", "Knowledge base:"]},
    {"id": "ev-weekly-plan", "question": "Find the best day and time to charge my EV this week, considering cost and solar.", "expected_tools": ["optimize_device_schedule_this_week", *forecast_tools], "expected_response": ["EV Charger", "estimated savings", "forecast solar", "Knowledge base:"]},
]
advisor = build_energy_advisor()

In [ ]:
test_results = []
for scenario in scenarios:
    inputs = {"question": scenario["question"], **scenario.get("inputs", {})}
    result = advisor.invoke(inputs)
    response = result["answer"]
    context = result.get("context", {})
    completed_tools = [call["name"] for call in context.get("tool_calls", []) if call["status"] == "completed"]
    response_checks = scenario["expected_response"]
    matched_checks = [check for check in response_checks if check.casefold() in response.casefold()]
    expected_tools = set(scenario["expected_tools"])
    actual_tools = set(completed_tools)
    missing_tools = sorted(expected_tools - actual_tools)
    unexpected_tools = sorted(actual_tools - expected_tools)
    tool_union = expected_tools | actual_tools
    test_results.append({
        "id": scenario["id"],
        "question": scenario["question"],
        "expected_tools": scenario["expected_tools"],
        "expected_response": response_checks,
        "decision": {"intent": result.get("intent"), "device": result.get("device"), "target_date": str(result.get("target_date"))},
        "tool_calls": context.get("tool_calls", []),
        "response": response,
        "scores": {
            "response_quality": round(100 * len(matched_checks) / len(response_checks), 1) if response_checks else 100.0,
            "tool_usage": round(100 * len(expected_tools & actual_tools) / len(tool_union), 1) if tool_union else 100.0,
        },
        "explanations": {
            "response_quality": f"Matched {len(matched_checks)} of {len(response_checks)} response checks: {matched_checks}.",
            "tool_usage": f"Missing expected tools: {missing_tools}; unexpected tools: {unexpected_tools}.",
        },
    })
print(f"Evaluated {len(test_results)} structured scenarios.")


QUESTION: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
EcoHome Energy Advisor | Sunday, October 4, 2026

Run the EV Charger from 10:00 to 14:00. Estimated energy cost: $1.75; estimated savings: $6.71 versus an 18:00 start at the same duration.

The estimate uses 13.1 kWh of forecast solar and 10.9 kWh of grid energy. Estimated solar-related avoided emissions: 5.2 kg CO2e (0.4 kg CO2e/kWh assumption).

Forecast: clear, 26% average cloud cover, about 26.5 kWh of daylight solar.

I couldn't find a scheduleable device in the question. Ask about an EV, dishwasher, pool pump, HVAC, or washer, or request a usage-history review.

Knowledge base: [tip_energy_storage.txt], [tip_renewable_integration.txt], [tip_energy_savings.txt]

Estimates use the bundled sample tariff and forecast data, not live utility or weather feeds.
------------------------------------------------------------------------

QUESTION: What temperature should I set my thermostat on

: 

In [3]:
average_response = sum(item["scores"]["response_quality"] for item in test_results) / len(test_results)
average_tool_usage = sum(item["scores"]["tool_usage"] for item in test_results) / len(test_results)
pass_rate = 100 * sum(item["scores"]["response_quality"] >= 80 and item["scores"]["tool_usage"] >= 80 for item in test_results) / len(test_results)
strengths = [item for item in test_results if item["scores"]["response_quality"] == 100 and item["scores"]["tool_usage"] == 100]
weaknesses = [item for item in test_results if item["scores"]["response_quality"] < 100 or item["scores"]["tool_usage"] < 100]
recommendations = []
if any(item["scores"]["response_quality"] < 100 for item in test_results):
    recommendations.append("Review responses that missed expected answer checks; improve intent routing or grounding for those scenarios.")
if any(item["scores"]["tool_usage"] < 100 for item in test_results):
    recommendations.append("Align tool selection with the scenario requirements and avoid unnecessary tool calls.")
if not recommendations:
    recommendations.append("All defined checks passed; add edge cases and user-specific constraints to broaden coverage.")
rows = "\n".join(f"| {item['id']} | {item['scores']['response_quality']:.1f}% | {item['scores']['tool_usage']:.1f}% |" for item in test_results)
report = f"""# Energy Advisor evaluation report

**Overall metrics**  
Scenarios: {len(test_results)}  
Average response quality: {average_response:.1f}%  
Average tool usage (Jaccard): {average_tool_usage:.1f}%  
Pass rate (both scores >= 80%): {pass_rate:.1f}%

| Scenario | Response quality | Tool usage |
|---|---:|---:|
{rows}

## Strengths
{chr(10).join('- ' + item['id'] + ': all response and tool checks passed.' for item in strengths) if strengths else '- No scenario achieved a perfect score; use the itemized explanations below to target improvements.'}

## Weaknesses
{chr(10).join('- ' + item['id'] + ': ' + item['explanations']['response_quality'] + ' ' + item['explanations']['tool_usage'] for item in weaknesses) if weaknesses else '- No measured weaknesses against the defined checks.'}

## Improvement recommendations
{chr(10).join('- ' + recommendation for recommendation in recommendations)}
"""
display(Markdown(report))

EcoHome Energy Advisor | Sunday, October 4, 2026

Run the EV Charger from 10:00 to 14:00. Estimated energy cost: $1.75; estimated savings: $6.71 versus an 18:00 start at the same duration.

The estimate uses 13.1 kWh of forecast solar and 10.9 kWh of grid energy. Estimated solar-related avoided emissions: 5.2 kg CO2e (0.4 kg CO2e/kWh assumption).

Forecast: clear, 26% average cloud cover, about 26.5 kWh of daylight solar.

I couldn't find a scheduleable device in the question. Ask about an EV, dishwasher, pool pump, HVAC, or washer, or request a usage-history review.

Knowledge base: [tip_energy_storage.txt], [tip_renewable_integration.txt], [tip_energy_savings.txt]

Estimates use the bundled sample tariff and forecast data, not live utility or weather feeds.


## Evaluation notes
Check that each response names a schedule or actionable strategy, states the cost baseline for any savings estimate, uses a cited knowledge document, and distinguishes seeded sample forecasts from live utility/weather data. Compare the recommendation against the hourly tariff and solar values in `data/energy_data.db`.